In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd
import numpy as np


from pathlib import Path

In [ ]:
DATA_DIR = Path("/content/drive/MyDrive/CciaDatosAva - TEC")
FILE_NAME = "UNE_TUNE_SEX_AGE_NB_A-20260202T2307.csv"
path = DATA_DIR / FILE_NAME

Fuente de datos:
`https://ilostat.ilo.org/es/data/`

Unemployment by sex and age (thousands)|Annual|Labour Force Statistics (LFS)



---



# Exploración Inicial -- Conociendo un conjunto de datos

Importemos un conjunto de datos y revisemos algunos métodos útiles de pandas para la exploración inicial

In [ ]:
desempleo = pd.read_csv(path, low_memory=False)

In [ ]:
print("Filas:", desempleo.shape[0])
print("Columnas:", desempleo.shape[1])

### `.head()` -- ¿Qué columnas tienen los datos y qué parecen representar?

In [ ]:
desempleo.head()

### Antes de continuar profundicemos un poco sobre los datos

| Columna                | Descripción                                                                     | Ejemplo                                    | Notas                                         |
| ---------------------- | ------------------------------------------------------------------------------- | ------------------------------------------ | --------------------------------------------- |
| `ref_area.label`       | País o área geográfica de referencia.                                           | `Chile`                                    | Es la unidad territorial del dato.            |
| `source.label`         | Fuente estadística / encuesta usada (LFS específica).                           | `LFS - Current Population Survey`          | Puede variar por país y por año.              |
| `indicator.label`      | Nombre del indicador.                                                           | `Unemployment by sex and age (thousands)`  | En este dataset es fijo.                      |
| `sex.label`            | Sexo del grupo observado.                                                       | `Female`, `Male`, `Total`                  | “Total” = ambos sexos combinados.             |
| `classif1.label`       | Clasificación 1 (en este dataset: grupo de edad).                               | `Age (10-year bands): 35-44`               | OJO: hay varios esquemas de edad.             |
| `time`                 | Año de referencia.                                                              | `2024`                                     | Frecuencia anual (`_A`).                      |
| `obs_value`            | Valor observado del indicador.                                                  | `1086.235`                                 | Unidad: **miles de personas** desempleadas.   |
| `obs_status.label`     | Bandera/estatus de la observación (calidad o comparabilidad).                   | `Break in series`                          | Si hay ruptura metodológica/serie.            |
| `note_classif.label`   | Nota asociada a la clasificación (edad).                                        | `Nonstandard age group: Excluding age 15`  | Sirve para detectar definiciones no estándar. |
| `note_indicator.label` | Nota asociada al indicador.                                                     | `Break in series: Methodology revised`     | Explica cambios del indicador.                |
| `note_source.label`    | Nota asociada a la fuente (metodología, cobertura, edad mínima, periodo, etc.). | `Age coverage - minimum age: 16 years old` | Súper útil para comparabilidad internacional. |


In [ ]:
desempleo['obs_status.label'].unique()


### `.dtypes` -- ¿Qué tipo de datos contiene el conjunto de datos?

In [ ]:
desempleo.dtypes

En pandas, que una columna tenga `dtype = object` significa básicamente que:

la columna contiene “objetos de Python”, y en la práctica casi siempre quiere decir que son strings (texto).

### `.info()` -- ¿Qué tan completo está el conjunto de datos y qué tipo de datos contiene?

In [ ]:
desempleo.info()

### `.isna()` -- ¿Qué valores tenemos en columnas categóricas?

In [ ]:
desempleo.isna().sum().sort_values(ascending=False).head(10)

### `.unique()` -- ¿Qué valores tenemos en columnas categóricas?

In [ ]:
desempleo['sex.label'].unique()

### `.value_counts()` -- ¿Cuántas valores tenemos en cada categoría en una columna categórica?

In [ ]:
desempleo.value_counts("sex.label")

### `.describe()` -- ¿Qué podemos decir rápidamente de los datos en las columnas numéricas?

In [ ]:
desempleo.describe()

### Histogramas

Los histogramas son una forma clásica de analizar la distribución de datos numéricos: dividen los valores en intervalos (bins) y muestran cuántos datos caen en cada uno.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

data = desempleo[desempleo['time'] == 2025]

# binwidth controla la resolución del histograma,
# ajustando qué tan finos o gruesos son los intervalos de agrupación
sns.histplot(data=data, x='obs_value', binwidth=100)

plt.show()

# Validación de datos

La validación de datos es un paso inicial importante en el análisis exploratorio de datos (EDA).

Queremos entender si los tipos de datos y sus rangos son los esperados antes de avanzar demasiado en nuestro análisis.

## Actualización de tipos de datos

In [ ]:
desempleo['sex.label'] = desempleo['sex.label'].astype('string')

In [ ]:
desempleo.dtypes

## Validando datos categóricos

In [ ]:
desempleo['sex.label'].isin(["Female", "Male"])

In [ ]:
~desempleo['sex.label'].isin(["Female", "Male"])

## Validando datos numéricos

## .select_dtypes("number")

Podemos seleccionar y visualizar únicamente las columnas numéricas de un DataFrame.

In [ ]:
desempleo.select_dtypes("number")

### `.min` y  `.max`

In [ ]:
print(data['obs_value'].max())

print(data['obs_value'].min())

In [ ]:
sns.boxplot(data=data, x='obs_value')
plt.show()

In [ ]:
data.obs_value.describe()

# Agregación de datos

### .groupby()

Podemos explorar con mayor detalle las características de subconjuntos de datos con ayuda de la función .groupby(), que agrupa los datos según una categoría determinada, permitiendo al usuario encadenar una función de agregación como .mean() o .count() para describir los datos dentro de cada grupo.

In [ ]:
data['classif1.label'].unique()

In [ ]:
data_slice2 = data[data['classif1.label'].isin([
    'Age (Youth, adults): 15-24',
    'Age (Youth, adults): 25+'
])]


In [ ]:
data_slice3 = data_slice2[['ref_area.label', 'obs_value', 'classif1.label', 'sex.label']]
data_slice4 = data_slice3[data_slice3['sex.label'] == 'Total']
data_slice4


In [ ]:
# Total de desempleados (miles) en 2025 por país, sumando jóvenes y adultos.
data_slice4.groupby('ref_area.label')['obs_value'].sum()

Otras funciones de agregación (Pandas)

- **Media:** `.mean()`
- **Conteo:** `.count()`
- **Mínimo:** `.min()`
- **Máximo:** `.max()`
- **Varianza:** `.var()`
- **Desviación estándar:** `.std()`


La función `.agg`, nos permite aplicar funciones de agregación. De forma predeterminada, agrega los datos considerando todas las filas de una columna determinada y normalmente se utiliza cuando queremos aplicar más de una función.

### Otros ejemplos de agregación

```python
df_movies.groupby("genre").agg(
    mean_rating=("rating", "mean"),
    std_rating=("rating", "std"),
    median_year=("year", "median")
)
```
| genre        | mean_rating | std_rating | median_year |
|--------------|-------------|------------|-------------|
| Fiction      | 4.570229    | 0.281123   | 2013.0      |
| Non Fiction  | 4.598324    | 0.179411   | 2013.0      |



```python
df_movies.agg({"rating": ["mean", "std"], "year": ["median"]})
```

|        | rating   | year   |
|--------|----------|--------|
| mean   | 4.608571 | NaN    |
| std    | 0.226941 | NaN    |
| median | NaN      | 2013.0 |


# Limpieza de datos e imputación

### Datos faltantes
**¿Por qué es importante tratar los datos faltantes?**

Porque pueden distorsionar las distribuciones y sesgar nuestras conclusiones.

Por ejemplo, imaginemos que queremos estudiar el abandono escolar en una base de estudiantes. Si no tenemos registros de abandono justamente para estudiantes con mayor riesgo (por ejemplo, quienes trabajan, tienen bajo promedio o faltan mucho), entonces el abandono promedio observado será artificialmente más bajo que el abandono real.

En otras palabras: el dataset deja de ser representativo de la población. Si algunos grupos están subrepresentados, podemos concluir erróneamente que el abandono escolar es menor de lo que realmente es.

In [ ]:
FILE_NAME_1 = "students_abandono_sintetico.csv"
path_1 = DATA_DIR / FILE_NAME_1

abandono = pd.read_csv(path_1, low_memory=False)

In [ ]:
abandono.sample(5)

## Diccionario de columnas (`abandono`)

* **`student_id`**: Identificador único del estudiante (ID).
* **`year`**: Año del registro/observación (ej. 2023–2025).
* **`school`**: Nombre o tipo de escuela donde está inscrito el estudiante.
* **`grade`**: Grado escolar (1, 2 o 3).
* **`sex`**: Sexo del estudiante (`Female` / `Male`).
* **`age`**: Edad del estudiante (en años).
* **`avg_grade`**: Promedio académico del estudiante (escala 0–10).
* **`absences`**: Número de faltas (inasistencias) acumuladas en el periodo.
* **`works`**: Si el estudiante trabaja además de estudiar (`Yes` / `No`).
* **`distance_km`**: Distancia aproximada de la casa a la escuela (en kilómetros).
* **`income_level`**: Nivel socioeconómico del hogar (`Low`, `Medium`, `High`).
* **`dropout`**: Variable objetivo: abandono escolar

  * `1` = abandonó
  * `0` = no abandonó
  * `NaN` = dato faltante (intencional para practicar imputación)

---

Si quieres, también te puedo dar una versión tipo “para reporte” en una tabla formal.


In [ ]:
# Revisar valores faltantes por columna

abandono.isna().sum()

In [ ]:
# Tasa de abandono por si trabaja
abandono.groupby("works")["dropout"].mean()

En este dataset, los estudiantes que trabajan tienen una tasa de abandono mucho mayor.

De hecho, quienes trabajan abandonan ~5.6 veces más que quienes no trabajan.

Estrategias para tratar missing data

Hay varias estrategias:

* Eliminar filas si los faltantes son pocos (regla común: <= 5%)
* Imputar con estadísticos (moda/mediana/promedio)
* Imputar por subgrupos (más realista) **


** Imputar por subgrupos significa que, si falta el dato de abandono, lo reemplazamos usando la tasa promedio de abandono de estudiantes con características similares, por ejemplo del grupo que sí trabaja.

Antes de imputar, necesitamos decidir qué tan grave es el problema de datos faltantes. Una regla práctica es eliminar registros solo si los valores faltantes representan una proporción pequeña, por ejemplo, el 5% o menos.

### Umbral para eliminar valores faltantes

Antes de imputar, necesitamos decidir qué tan grave es el problema de datos faltantes. Una regla práctica es eliminar registros solo si los valores faltantes representan una proporción pequeña, por ejemplo, el 5% o menos.

In [ ]:
threshold = len(abandono) * 0.05
threshold

### Eliminación de valores faltantes (drop)

Ahora identificamos las columnas con pocos valores faltantes y eliminamos únicamente las filas donde esas variables están incompletas. Esto reduce el ruido sin perder demasiada información.

In [ ]:
missing_by_col = abandono.isna().sum()

cols_to_drop = missing_by_col[(missing_by_col > 0) & (missing_by_col <= threshold)].index.tolist()
cols_to_drop

In [ ]:
# Ahora eliminamos filas con NA en esas columnas
abandono.dropna(subset=cols_to_drop, inplace=True)

### Imputación con estadísticos (modo/mediana)

Para las columnas restantes, en lugar de eliminar más datos, rellenamos los valores faltantes usando estadísticas simples: la moda para variables categóricas y la mediana para variables numéricas.

In [ ]:
# imputación por moda (categorías)
for col in ["avg_grade", "distance_km"]:
    abandono[col] = abandono[col].fillna(abandono[col].mode()[0])

# imputación por mediana (numéricas)
for col in ["avg_grade", "distance_km"]:
    abandono[col] = abandono[col].fillna(abandono[col].median())

### Verificación de valores faltantes restantes

Después de eliminar e imputar, verificamos nuevamente cuántos valores faltantes quedan. Este paso es clave para confirmar que nuestras decisiones realmente mejoraron la calidad del dataset.

In [ ]:
abandono.isna().sum()

### Imputación por subgrupos

Sin embargo, en variables importantes como el abandono escolar, usar un valor global puede ser poco realista. Por eso, imputamos por subgrupos: asignamos valores basados en estudiantes con características similares, como si trabajan o no.

### Aplicación de imputación por subgrupos

Con las tasas de abandono calculadas por grupo, imputamos los valores faltantes asignando a cada estudiante la tasa correspondiente a su subgrupo, conservando patrones reales del fenómeno.

In [ ]:
# En abandono escolar, imputar dropout con promedio global suele ser mala idea.
# Mejor imputar por grupos de riesgo, por ejemplo por grade y works
# Entonces calculamos el abandono típico para cada combinación de grado y condición de trabajo,
# y lo guarda como referencia para rellenar valores faltantes

dropout_rate = (
    abandono
    .groupby(["grade", "works"])["dropout"]
    .median()
    .to_dict()
)

dropout_rate


In [ ]:
# Identificamos qué estudiantes NO tienen registrado si abandonaron o no (dropout vacío / NaN)
mask = abandono["dropout"].isna()

# Para esos estudiantes con dropout faltante:
# Vamos a completar el valor usando la "tasa típica" de abandono de su grupo.
# El grupo se define por:
# - su grado escolar (grade)
# - si trabaja o no (works)

abandono.loc[mask, "dropout"] = abandono.loc[mask, ["grade", "works"]].apply(
    lambda x: dropout_rate.get((x["grade"], x["works"]), np.nan),
    axis=1
)

# Si todavía quedan valores vacíos (porque algún grupo no tenía datos suficientes),
#    rellenamos esos casos con un valor general del dataset:
#    la mediana del abandono (una medida "típica" global).
abandono["dropout"] = abandono["dropout"].fillna(abandono["dropout"].median())


### Dataset final sin faltantes

Finalmente, confirmamos que ya no hay valores faltantes. Ahora el dataset está listo para análisis y modelos sin el sesgo o los errores que pueden generar los datos incompletos.

In [ ]:
abandono.isna().sum()

# Datos categóricos

Ahora vamos a explorar cómo crear y analizar variables categóricas en un para entender mejor el abandono escolar.

Veamos cómo:
* Identificar columnas categóricas,
* Contar frecuencias,
* Extraer información útil desde texto (por ejemplo tipos de escuela), y
* Crear una nueva columna categórica que agrupe escuelas en categorías más útiles para el análisis.

In [ ]:
# ============================================================
# Identificar columnas categóricas
# ============================================================

# Seleccionamos columnas NO numéricas (categóricas/texto)
cat_cols = abandono.select_dtypes(exclude="number")

# Vemos algunas filas de esas columnas
cat_cols.sample(5)

In [ ]:
# ============================================================
# Contar frecuencia de valores
# ============================================================

# Contamos cuántos estudiantes hay por tipo de escuela
abandono["school"].value_counts()

In [ ]:
# ============================================================
# Cuántas escuelas distintas hay?
# ============================================================

abandono["school"].nunique()

In [ ]:
# ============================================================
# Extraer valor desde categorías (texto)
# Ejemplo: identificar si la escuela es "Telesecundaria"
# ============================================================

# Crea una serie booleana: True si la palabra aparece en el texto
is_telesec = abandono["school"].str.contains("Telesecundaria", case=False, na=False)

# Filtramos solo estudiantes de telesecundaria
abandono[is_telesec].head()


In [ ]:
# ============================================================
# Buscar múltiples frases: "Técnica" o "Telesecundaria"
# ============================================================

# El símbolo | significa OR (una cosa u otra)
is_special_school = abandono["school"].str.contains("Técnica|Telesecundaria", case=False, na=False)

abandono.loc[is_special_school, "school"].value_counts()

In [ ]:
# ============================================================
# Buscar escuelas que empiezan con "Secundaria"
# ============================================================

# ^ indica inicio del texto
starts_secundaria = abandono["school"].str.contains("^Secundaria", case=False, na=False)

abandono.loc[starts_secundaria, "school"].value_counts()

In [ ]:
# ============================================================
# Crear una nueva columna categórica: school_type
# ============================================================

# Definimos condiciones (cada condición es un filtro)
cond_telesec = abandono["school"].str.contains("Telesecundaria", case=False, na=False)
cond_tecnica = abandono["school"].str.contains("Técnica", case=False, na=False)
cond_secundaria = abandono["school"].str.contains("^Secundaria", case=False, na=False)

# case=False: no distingue mayúsculas/minúsculas
# na=False: los vacíos cuentan como no cumple


# Definimos etiquetas para cada condición
choices = ["Telesecundaria", "Secundaria Técnica", "Secundaria General"]

# np.select asigna una categoría según la primera condición que se cumpla
abandono["school_type"] = np.select(
    [cond_telesec, cond_tecnica, cond_secundaria],
    choices,
    default="Other"   # si no cumple ninguna condición, se asigna Other
)

# Si la escuela contiene 'Telesecundaria' → 'Telesecundaria'
# Si no, pero contiene 'Técnica' → pon 'Secundaria Técnica'
# Si no, pero empieza con 'Secundaria' → pon 'Secundaria General'
# Si no cumple ninguna → 'Other'

# Revisamos rápidamente que quedó bien
abandono[["school", "school_type"]].head()

In [ ]:
# ============================================================
# Analizar abandono por categoría creada
# ============================================================

# Promedio de dropout (0/1) = tasa de abandono
dropout_by_schooltype = abandono.groupby("school_type")["dropout"].mean().sort_values(ascending=False)

dropout_by_schooltype


In [ ]:
# ============================================================
# Visualizar: tasa de abandono por tipo de escuela
# ============================================================

(dropout_by_schooltype * 100).plot(kind="bar")
plt.ylabel("Tasa de abandono (%)")
plt.title("Abandono escolar por tipo de escuela")
plt.show()


# Datos numéricos

Ahora vamos a cambiar el enfoque a trabajar con datos numéricos. En particular, veremos cómo:

* Convertir una columna numérica que viene como texto (por ejemplo, promedio con coma decimal),
* Crear una nueva variable numérica útil, y
* Agregar estadísticos por grupo directamente al DataFrame, por ejemplo:
la desviación estándar del promedio por grado, o la mediana de faltas por nivel socioeconómico.

In [ ]:
# ============================================================
# Trabajando con datos numéricos: revisar dataset original
# ============================================================

# Resumen del DataFrame: columnas, tipos y valores no nulos
print(abandono.info())

print("###############################")

# Estadísticas rápidas de columnas numéricas
print(abandono.describe())

In [ ]:
# ============================================================
# Ejemplo: columna numérica que viene como texto
#    (simulamos que avg_grade viene como string con coma decimal)
# ============================================================

# Creamos una copia para no modificar el original
df = abandono.copy()

# Convertimos avg_grade a texto y le ponemos coma decimal como ejemplo ("8,5")
df["avg_grade_str"] = df["avg_grade"].astype(str).str.replace(".", ",", regex=False)

# Revisamos cómo se ve
display(df[["avg_grade", "avg_grade_str"]].head())

print("#########")

df["avg_grade_str"].info()


In [ ]:
# ============================================================
# Convertir strings a números
# ============================================================

# Paso 1: reemplazar coma por punto para poder convertirlo a número
df["avg_grade_clean"] = df["avg_grade_str"].str.replace(",", ".", regex=False)

# Paso 2: convertir a float
df["avg_grade_clean"] = pd.to_numeric(df["avg_grade_clean"], errors="coerce")

# errors="coerce": Si no se puede convertir a número, no marques error: pon NaN (vacío)

# Verificamos el resultado
df[["avg_grade_str", "avg_grade_clean"]].head()

In [ ]:
# ============================================================
# Crear una nueva variable numérica útil para abandono
#    Ejemplo: "absences_rate" = faltas por mes (aprox.)
# ============================================================

# Suponiendo que el periodo observado es de ~10 meses
df["absences_rate"] = df["absences"] / 10

df[["absences", "absences_rate"]].head()

In [ ]:
# ============================================================
# Estadísticos por grupo en tabla (resumen)
#    Ejemplo: promedio de abandono por nivel socioeconómico
# ============================================================

df.groupby("income_level")["dropout"].mean()

El abandono es mucho mayor en estudiantes de ingreso bajo. Los estudiantes de ingreso bajo tienen una tasa de abandono ~4 veces mayor que los de ingreso alto.

# Valores extremos / datos atípicos

Ahora veamos cómo detectar y manejar .

Un **valor extremo** es un valor que está muy lejos del resto. En este contexto, por ejemplo:

* un estudiante con 200 faltas cuando la mayoría tiene menos de 30, o
* una distancia a la escuela de 150 km cuando casi todos viven a menos de 15 km.

Estos valores extremos pueden deberse a:

* errores de captura,
* unidades incorrectas,
* o casos reales pero poco comunes.

Detectarlos es importante porque pueden distorsionar el promedio, inflar la desviación estándar y afectar modelos de predicción del abandono.

In [ ]:
# ============================================================
# Manejo de outliers
# ============================================================

# Revisamos estadísticos descriptivos de faltas
abandono["absences"].describe()

# Si el máximo es demasiado alto comparado con el percentil 75,
# probablemente hay valores extremos (outliers).

### IQR

El **rango intercuartil (IQR)** es una forma muy usada para medir qué tan dispersos están los datos y para detectar valores extremos.

In [ ]:
# ============================================================
# Calcular IQR (Interquartile Range)
#    IQR = Q3 - Q1
# ============================================================

Q3 = abandono["absences"].quantile(0.75)  # percentil 75
Q1 = abandono["absences"].quantile(0.25)  # percentil 25
IQR = Q3 - Q1

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)

In [ ]:
3/4

Q1: 6.0  (Cuartil 1 / Percentil 25)

El **25% de los estudiantes** tiene **6 faltas o menos**.

Dicho de otra forma: 1 de cada 4 estudiantes tiene como máximo 6 faltas. (1/4 = 0.25)

---
Q3: 11.0 (Cuartil 3 / Percentil 75)

El **75% de los estudiantes** tiene **11 faltas o menos**.

Dicho de otra forma: 3 de cada 4 estudiantes tiene como máximo 11 faltas. (1/4 = 0.75)


---

IQR: 5.0 (Rango intercuartil)

El **50% central** de los estudiantes está entre:

* Q1 = 6 faltas
* Q3 = 11 faltas

Esto significa que la mitad “más típica” del grupo tiene faltas en un rango de **5 faltas** (de 6 a 11).

### Valor extremo definición precisa

Ahora que ya calculamos el rango intercuartil (IQR), podemos usarlo para definir de forma objetiva qué valores se consideran “extremos”. Para esto aplicamos la regla estándar 1.5 × IQR, que establece un límite superior y un límite inferior. Cualquier observación por encima del límite superior o por debajo del límite inferior se considera un posible outlier.

In [ ]:
# ============================================================
# Definir umbrales de outliers con regla 1.5*IQR
#    Outlier alto: > Q3 + 1.5*IQR
#    Outlier bajo: < Q1 - 1.5*IQR
# ============================================================

upper_limit = Q3 + 1.5 * IQR
lower_limit = Q1 - 1.5 * IQR

print("Lower limit:", lower_limit)
print("Upper limit:", upper_limit)

# Nota: en faltas, lower_limit podría salir negativo.
# Eso no es un problema: simplemente significa que no hay outliers bajos.

Usando la regla 1.5×IQR, se considera valor extremo a cualquier estudiante con 19 o más faltas, mientras que no hay valores extremos por debajo porque el límite inferior es negativo.

In [ ]:
# ============================================================
# Identificar outliers (filtrar estudiantes con faltas extremas)
# ============================================================

outliers_absences = abandono[
    (abandono["absences"] < lower_limit) | (abandono["absences"] > upper_limit)
]

# Mostramos algunos casos extremos con variables útiles
outliers_absences[["student_id", "grade", "works", "income_level", "absences", "dropout"]].head(10)

In [ ]:
len(outliers_absences)

Una vez identificados los valores extremos, es importante evaluar qué tanto afectan el análisis. Para hacerlo, comparamos estadísticas clave (como la media y el valor máximo) antes y después de eliminar los outliers. Esto nos permite ver si los valores extremos están distorsionando los resultados y si conviene tratarlos o no.

In [ ]:
# ============================================================
# ¿Por qué importa? Comparar estadísticos con y sin outliers
# ============================================================

print("Media original:", abandono["absences"].mean())
print("Max original:", abandono["absences"].max())

# Quitamos outliers (solo conservamos valores dentro de los límites)
no_outliers = abandono[
    (abandono["absences"] >= lower_limit) & (abandono["absences"] <= upper_limit)
]

print("Media sin outliers:", no_outliers["absences"].mean())
print("Max sin outliers:", no_outliers["absences"].max())

In [ ]:
# ============================================================
# Visualizar impacto: histogramas
# ============================================================

# Histograma con outliers
abandono["absences"].plot(kind="hist", bins=30)
plt.title("Distribución de faltas (con outliers)")
plt.xlabel("Faltas")
plt.show()

# Histograma sin outliers
no_outliers["absences"].plot(kind="hist", bins=30)
plt.title("Distribución de faltas (sin outliers)")
plt.xlabel("Faltas")
plt.show()

Los valores extremos hacen que la distribución se vea más sesgada (con cola larga), aumentan el rango y pueden inflar estadísticas como la media y la desviación estándar.

In [ ]:
# ============================================================
# Visualizar impacto mejorado: histogramas (bins enteros, sin huecos)
# ============================================================

# Definimos bins centrados en enteros:
# -0.5, 0.5, 1.5, 2.5, ... para que cada barra represente un número entero de faltas
min_abs = int(abandono["absences"].min())
max_abs = int(abandono["absences"].max())

bins_all = np.arange(min_abs - 0.5, max_abs + 1.5, 1)

# Histograma con outliers
plt.figure()
plt.hist(abandono["absences"].dropna(), bins=bins_all)
plt.title("Distribución de faltas (con outliers)")
plt.xlabel("Faltas")
plt.ylabel("Frecuencia")
plt.xticks(range(min_abs, max_abs + 1, 2))  # ticks cada 2 para que no se amontonen
plt.show()


# Ahora bins para el dataset sin outliers (mismo estilo)
min_no = int(no_outliers["absences"].min())
max_no = int(no_outliers["absences"].max())

bins_no = np.arange(min_no - 0.5, max_no + 1.5, 1)

# Histograma sin outliers
plt.figure()
plt.hist(no_outliers["absences"].dropna(), bins=bins_no)
plt.title("Distribución de faltas (sin outliers)")
plt.xlabel("Faltas")
plt.ylabel("Frecuencia")
plt.xticks(range(min_no, max_no + 1, 2))
plt.show()


In [ ]:
# ============================================================
# Qué hacer con outliers (decisión)
# ============================================================

# Preguntas clave:
# - ¿Son valores reales? (casos extremos pero válidos)
# - ¿O son errores? (ej: faltas registradas mal, unidades mal)
#
# Si son errores, se pueden:
# - eliminar (drop)
# - recortar (winsorizar: poner un máximo permitido)
#
# Ejemplo de winsorización (cap):
abandono["absences_capped"] = abandono["absences"].clip(upper=upper_limit)

abandono[["absences", "absences_capped"]].sample(5)

